# Tutorial 07: GPU Backend Benchmark — CuPy vs PyTorch vs Fused CUDA Kernels

LPF solvers automatically detect the model's device and select the optimal execution path:

| Model device | Internal behavior |
|---|---|
| `"cpu"` | NumPy array operations (Python loop) |
| `"cuda:0"` | CuPy arrays → **fused CUDA kernels (automatic)** |
| `"torch:cuda:0"` | PyTorch tensors → **DLPack zero-copy bridge → same CUDA kernels** |

No manual `backend=` option is needed.
Just use `EulerSolver()` as usual — the solver inspects `model.am` and dispatches accordingly.

### How do PyTorch tensors reach the CUDA kernels?

```
EulerSolver.solve()
  └─ _is_cuda(model) → True (TorchModule + cuda)
     └─ CuSolverBase.solve()
        └─ _maybe_bridge_torch(model)
           └─ cp.from_dlpack(tensor.detach())  ← zero-copy!
              Shares GPU memory; no data transfer.
        └─ Execute CUDA kernels (CuPy RawKernel)
        └─ _restore_torch(model, saved)
           └─ Restore original PyTorch tensor references
              (in-place modifications are already visible)
```

In this tutorial we simulate 16 ladybird wing-pattern models in batch on GPU
and compare the wall-clock performance across all three paths.

In [1]:
import os
import os.path as osp
from os.path import join as pjoin
import time

import numpy as np
from PIL import Image

from lpf.initializers import LiawInitializer
from lpf.models import LiawModel
from lpf.solvers import EulerSolver, HeunSolver, RungeKuttaSolver
from lpf.data import load_model_dicts

## 1. Simulation Parameters

In [ ]:
dt = 0.01
n_iters = 500_000       # For benchmarking (full run: 500,000)
dx = 0.1
width = 128
height = 128

# Load 16 model parameter sets (batch_size=16)
LPF_REPO_HOME = osp.abspath("..")
dpath_pop = pjoin(LPF_REPO_HOME, "population", "init_pop_axyridis")
model_dicts = load_model_dicts(dpath_pop)

print(f"Batch size: {len(model_dicts)} models")
print(f"Grid: {height} x {width}, dt={dt}, n_iters={n_iters:,}")

Batch size: 16 models
Grid: 128 x 128, dt=0.01, n_iters=50,000


## 2. Helper Functions

In [3]:
def create_model(device):
    """Create a Liaw model on the given device."""
    initializer = LiawInitializer()
    initializer.update(model_dicts)
    params = LiawModel.parse_params(model_dicts)
    return LiawModel(
        initializer=initializer,
        params=params,
        dx=dx, width=width, height=height,
        device=device,
    )


def sync_gpu():
    """Wait for all GPU work to finish."""
    try:
        import cupy as cp
        cp.cuda.Stream.null.synchronize()
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.synchronize()
    except Exception:
        pass


def benchmark(solver, model, label, n_iters=n_iters, n_warmup=200):
    """Warm up, then measure wall-clock time."""
    # Warm-up (JIT compilation, memory allocation, etc.)
    solver.solve(model=model, dt=dt, n_iters=n_warmup,
                 init_model=True, verbose=0)
    sync_gpu()

    # Benchmark
    model.initialize()
    sync_gpu()

    t_start = time.perf_counter()
    solver.solve(model=model, dt=dt, n_iters=n_iters,
                 init_model=False, verbose=0)
    sync_gpu()
    elapsed = time.perf_counter() - t_start

    print(f"  [{label}] {elapsed:.2f}s ({n_iters / elapsed:,.0f} iters/s)")
    return elapsed

## 3. Euler Solver — Three Devices Compared

The same `EulerSolver()` is used throughout.
The internal execution path changes based on the **model's device**:

- `device="cpu"` — NumPy array operations (Python loop)
- `device="cuda:0"` — CuPy model → fused CUDA kernel (automatic)
- `device="torch:cuda:0"` — PyTorch tensor → DLPack bridge → same CUDA kernel

In [ ]:
print(f"=== Euler ({len(model_dicts)} models, {n_iters:,} iters) ===")
print()

results = {}

# CPU (NumPy)
model_cpu = create_model("cpu")
results["Euler CPU"] = benchmark(EulerSolver(), model_cpu, "CPU (NumPy)")

# CuPy → automatic CUDA kernel
model_cupy = create_model("cuda:0")
results["Euler CuPy"] = benchmark(EulerSolver(), model_cupy, "CuPy -> CUDA kernel")

# PyTorch CUDA → DLPack bridge → same CUDA kernel
model_torch = create_model("torch:cuda:0")
results["Euler PyTorch"] = benchmark(EulerSolver(), model_torch, "PyTorch -> DLPack -> CUDA kernel")

=== Euler (16 models, 50,000 iters) ===



## 4. RK4 Solver — Same Comparison

The 4th-order Runge-Kutta method evaluates the PDE function 4 times per step.
More kernel launches per step make the GPU advantage even more pronounced.

In [ ]:
print(f"=== RK4 ({len(model_dicts)} models, {n_iters:,} iters) ===")
print()

# CPU
model_cpu = create_model("cpu")
results["RK4 CPU"] = benchmark(RungeKuttaSolver(), model_cpu, "CPU (NumPy)")

# CuPy
model_cupy = create_model("cuda:0")
results["RK4 CuPy"] = benchmark(RungeKuttaSolver(), model_cupy, "CuPy -> CUDA kernel")

# PyTorch
model_torch = create_model("torch:cuda:0")
results["RK4 PyTorch"] = benchmark(RungeKuttaSolver(), model_torch, "PyTorch -> DLPack -> CUDA kernel")

## 5. Results

In [ ]:
print(f"{'Method':<35} {'Time (s)':>10} {'vs CPU':>10}")
print("-" * 57)

for solver_name in ["Euler", "RK4"]:
    cpu_key = f"{solver_name} CPU"
    base = results[cpu_key]
    for suffix in ["CPU", "CuPy", "PyTorch"]:
        key = f"{solver_name} {suffix}"
        t = results[key]
        speedup = base / t
        print(f"{key:<35} {t:>10.2f} {speedup:>9.1f}x")
    print()

## 6. fast_math Option

Setting `fast_math=True` enables the `--use_fast_math` CUDA compiler flag,
which uses approximate math intrinsics for a small additional speedup.
Useful during exploratory search phases where slight precision loss is acceptable.

```python
# fast_math is passed to the solver and forwarded to the CUDA kernel compiler
solver = EulerSolver(fast_math=True)
```

In [ ]:
print("=== fast_math comparison (Euler, CuPy device) ===")
print()

model_fm = create_model("cuda:0")
t_normal = benchmark(EulerSolver(fast_math=False), model_fm, "fast_math=False")

model_fm = create_model("cuda:0")
t_fast = benchmark(EulerSolver(fast_math=True), model_fm, "fast_math=True")

print(f"\n  fast_math speedup: {t_normal / t_fast:.2f}x")

## 7. Visualizing the Batch Results

Visualize the 16 morphs computed via CUDA kernels in a 4x4 grid.

In [ ]:
model_vis = create_model("cuda:0")

t_beg = time.perf_counter()
EulerSolver().solve(model=model_vis, dt=dt, n_iters=n_iters, verbose=0)
sync_gpu()
print(f"Simulation: {time.perf_counter() - t_beg:.2f}s")

In [ ]:
from lpf.visualization import merge_multiple

arr_color = model_vis.colorize(thr_color=0.5)
imgs = [model_vis.create_image(i, arr_color)[0] for i in range(arr_color.shape[0])]

img_merged = merge_multiple(imgs=imgs, n_cols=4, ratio_resize=0.8, bg_color="white")
img_merged

## 8. Numerical Accuracy Verification

Since the CuPy and PyTorch paths use the exact same CUDA kernels
(via DLPack zero-copy), their results should match bit-for-bit.
The CPU path uses a different operation ordering, so minor floating-point
differences are expected.

In [ ]:
n_verify = 1000

# CuPy path
model_ref = create_model("cuda:0")
EulerSolver().solve(model=model_ref, dt=dt, n_iters=n_verify, verbose=0)
u_cupy = model_ref.am.get(model_ref.u)

# PyTorch path (DLPack bridge -> same CUDA kernel)
model_torch = create_model("torch:cuda:0")
EulerSolver().solve(model=model_torch, dt=dt, n_iters=n_verify, verbose=0)
u_torch = model_torch.am.get(model_torch.u)

# CPU path (reference — different operation ordering)
model_cpu = create_model("cpu")
EulerSolver().solve(model=model_cpu, dt=dt, n_iters=n_verify, verbose=0)
u_cpu = model_cpu.am.get(model_cpu.u)

print("CuPy vs PyTorch (same kernel, DLPack shared memory):")
print(f"  max |diff| = {np.max(np.abs(u_cupy - u_torch)):.2e}")
print(f"  allclose   = {np.allclose(u_cupy, u_torch)}")
print()
print("CuPy vs CPU (different execution path):")
print(f"  max |diff| = {np.max(np.abs(u_cupy - u_cpu)):.2e}")
print(f"  allclose   = {np.allclose(u_cupy, u_cpu, rtol=1e-4)}")

## Summary

### Usage — just change the device

```python
# CPU
model = LiawModel(..., device="cpu")

# GPU (CuPy) — automatically uses fused CUDA kernels
model = LiawModel(..., device="cuda:0")

# GPU (PyTorch) — DLPack bridge to the same CUDA kernels
model = LiawModel(..., device="torch:cuda:0")

# Solver API is always the same
solver = EulerSolver()                  # or HeunSolver(), RungeKuttaSolver()
solver = EulerSolver(fast_math=True)    # approximate math for extra speed
solver.solve(model=model, dt=0.01, n_iters=500000)
```

### Internal dispatch flow

```
solver.solve(model)
  |
  +-- model.am is CupyModule?
  |   Yes -> CuSolverBase.solve() -> launch CUDA kernels directly
  |
  +-- model.am is TorchModule + cuda?
  |   Yes -> CuSolverBase.solve()
  |          -> _maybe_bridge_torch(): DLPack zero-copy
  |          -> launch CUDA kernels
  |          -> _restore_torch(): restore PyTorch tensor refs
  |
  +-- otherwise -> default Python loop (NumPy / JAX / etc.)
```